<a href="https://colab.research.google.com/github/Saharochek98/Statistics-of-Stochastic-Processes/blob/main/05_nonparametric_sde_estimation_and_kernel_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy import stats
from scipy.stats import  ncx2, gamma
import pandas as pd
from numba import  njit
from typing import Literal
from scipy.integrate import simpson
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Examples

In [ ]:
class StochasticProcess:
    def __init__(self, params, T, Nx, Nt, init_state = None):
        self.params = params
        self.T = T
        self.Nx = Nx # Number of trajectories
        self.Nt = Nt # Number of discretization steps
        self.init_state = init_state

    #Wiener Process by default
    #use @staticmethod for numba compatibility with @njit
    @staticmethod
    @njit
    def bxt(x, t, params):
        return 0

    @staticmethod
    @njit
    def sigmaxt(x, t, params):
        return 1

    @staticmethod
    @njit
    def bxt_x(x, t, params):
        return 0

    @staticmethod
    @njit
    def bxt_xx(x, t, params):
        return 0

    @staticmethod
    @njit
    def sigmaxt_x(x, t, params):
        return 0

    @staticmethod
    @njit
    def sigmaxt_xx(x, t, params):
        return 0

    @staticmethod
    @njit
    def bxt_t(x, t, params):
        return 0

    def DefaultInitState(self):
        return np.zeros(self.Nx)

    def EulerSolution(self, dwt = None):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if dwt is None:
            dwt = np.random.normal(0, 1, size = (self.Nt, self.Nx)) * np.sqrt(dt)

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] + self.bxt(x_data[i - 1], t_data[i - 1], self.params) * dt +\
                        self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params) * dwt[i - 1]
        return t_data, x_data

    def Milstein2Solution(self, dwt = None):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if dwt is None:
            dwt = np.random.normal(0, 1, size = (self.Nt, self.Nx)) * np.sqrt(dt)

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] + (self.bxt(x_data[i - 1], t_data[i - 1], self.params) - 1/2 * self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params) * self.sigmaxt_x(x_data[i - 1], t_data[i - 1], self.params)) * dt +\
                        self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params) * dwt[i - 1] + 1/2 * self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params) * self.sigmaxt_x(x_data[i - 1], t_data[i - 1], self.params) * dwt[i - 1]**2 +\
                        (1/2 * self.bxt(x_data[i - 1], t_data[i - 1], self.params) * self.sigmaxt_x(x_data[i - 1], t_data[i - 1], self.params) + 1/2 * self.bxt_x(x_data[i - 1], t_data[i - 1], self.params) * self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params) +\
                        1/4 * self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params)**2 * self.sigmaxt_xx(x_data[i - 1], t_data[i - 1], self.params)) * dt * dwt[i - 1] +\
                        (1/2 * self.bxt(x_data[i - 1], t_data[i - 1], self.params) * self.bxt_x(x_data[i - 1], t_data[i - 1], self.params) + 1/4 * self.bxt_xx(x_data[i - 1], t_data[i - 1], self.params) * self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params)**2) * dt**2
        return t_data, x_data


class CIRprocess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

    # params = [theta1, theta2, theta3]
    @staticmethod
    @njit
    def bxt(x, t, params):
        return params[0] - params[1] * x

    @staticmethod
    @njit
    def sigmaxt(x, t, params):
        return params[2] * np.sqrt(np.maximum(x,0))

    @staticmethod
    @njit
    def bxt_x(x, t, params):
        return -params[1]

    @staticmethod
    @njit
    def sigmaxt_x(x, t, params):
        return params[2] * 1/2 * 1/np.sqrt(np.maximum(x,1e-12))

    @staticmethod
    @njit
    def sigmaxt_xx(x, t, params):
        return -params[2] * 1/4 * 1/np.maximum(x,1e-12)**(3/2)

    def DefaultInitState(self):
        return self.params[0] / self.params[1] * np.ones(self.Nx)

    def ExactSolution(self, dwt=None):
        """Exact CIR transitions; dwt is unsupported because this uses chi-square draws."""
        if dwt is not None:
            raise ValueError("Exact CIR transitions require noncentral chi-square draws, not Brownian increments.")
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.empty((self.Nt + 1, self.Nx))
        x_data[0] = self.DefaultInitState() if self.init_state is None else self.init_state
        theta1, theta2, theta3 = self.params
        if theta1 <= 0 or theta2 <= 0 or theta3 <= 0:
            raise ValueError("CIR parameters must be positive.")
        decay = np.exp(-theta2 * dt)
        scale = theta3**2 * (1 - decay) / (4 * theta2)
        degrees = 4 * theta1 / theta3**2
        for i in range(1, self.Nt + 1):
            noncentrality = 4 * theta2 * decay * np.maximum(x_data[i - 1], 0) / (theta3**2 * (1 - decay))
            x_data[i] = scale * ncx2.rvs(degrees, noncentrality, size=self.Nx)
        return t_data, x_data

    def StationaryState(self):
        theta1 = self.params[0]
        theta2 = self.params[1]
        theta3 = self.params[2]

        k = theta2
        theta = theta1 / theta2
        sigma = theta3

        c = 2 * k / (sigma**2)
        u = 0
        q = 2 * k * theta / sigma**2 - 1

        return ncx2.rvs(2 * q + 2, 2 * u, size = self.Nx) / (2 * c)

    def ProcessDistribution(self, x, t, x0, t0):
        theta1 = self.params[0]
        theta2 = self.params[1]
        theta3 = self.params[2]

        k = theta2
        theta = theta1 / theta2
        sigma = theta3

        c = 2 * k / (sigma**2 * (1 - np.exp(-k * (t - t0))))
        u = c * x0 * np.exp(-k * (t - t0))
        #v = c * x
        q = 2 * k * theta / sigma**2 - 1
        return ncx2.pdf(2 * c * x, 2 * q + 2, 2 * u) * (2 * c) #2c is normalazing constant

    def StationaryDistribution(self, x):
        theta1 = self.params[0]
        theta2 = self.params[1]
        theta3 = self.params[2]

        k = theta2
        theta = theta1 / theta2
        sigma = theta3

        c = 2 * k / (sigma**2)
        u = 0
        q = 2 * k * theta / sigma**2 - 1
        return ncx2.pdf(2 * c * x, 2 * q + 2, 2 * u) * (2 * c)

#Exercise 1. Simulate a Cox-Ingersoll-Ross (CIR) model
$$
dX_t = (\theta_1 - \theta_2 X_t)dt + \theta_3 \sqrt{X_t}dW_t
$$
with $\theta = (6, 2, 1)$, and choose the bandwidth according to Scott’s rule and implement the non-parametric kernel estimation for stationary density of the CIR model.

Plot in a graph the true stationary density against the estimated one for
*   $n=1000$, $\Delta_n=1$,
*   $n=15000$, $\Delta_n=0.01$.



In [ ]:
def StationaryDistributionKernelEstimator(x, x_data, delta = None):
    n = len(x_data)
    m = 1

    hn = 1

    if delta is None:
        hn = np.std(x_data) * n**(-1 / (m + 4))
    else:
        hn = delta

    pi = np.zeros(len(x))

    for j in range(0, len(x)):
        K = 1/np.sqrt(2 * np.pi) * np.exp(-1/2 * (x[j] - x_data)**2 / hn**2)

        pi[j] = 1 / (n * hn) * np.sum(K)

    return pi

In [ ]:
'''Single-trajectory calculation
Use a long time horizon to study stationarity. T/Nt is the time step.
'''

Nx = 1
Nt = 100000
T = 1000

params_CIR = [6, 2, 1]

CIRp = CIRprocess(params_CIR, T, Nx, Nt)
CIRp.init_state = CIRp.StationaryState()
t_data, x_data_CIR = CIRp.ExactSolution()

In [ ]:
n = np.random.randint(0, Nt)
fig, ax = plt.subplots()
x_data = np.linspace(0, 8, 250)

i = np.random.randint(0, Nt)

n, bins, patches = ax.hist(x_data_CIR.flatten(), bins=x_data, density=True, color='#0504aa', alpha=0.7, rwidth=0.85, label = 'SDE sample')
ax.set_xlabel('x')
ax.set_ylabel('p(x)')
ax.set_title(f'CIR-process', loc = 'center')

ax.plot(x_data, CIRp.StationaryDistribution(x_data), label = 'Stationary CIR PDF', color = 'red')
#ax.plot(x_data, CIRp.ProcessDistribution(x_data, T * i / Nt, x0, 0), label = 'Stationary CIR PDF', color = 'red')

ax.plot(x_data, StationaryDistributionKernelEstimator(x_data, x_data_CIR.flatten()), label = f'Kernel estimation. Scotts rule', color = 'green')

d = 0.01
ax.plot(x_data, StationaryDistributionKernelEstimator(x_data, x_data_CIR.flatten(), delta = d),
        label = f'Kernel estimation. delta = {d}', color = 'orange')

ax.legend(loc = 'upper right', fancybox = True, shadow = True)
plt.show()

In [ ]:
'''Calculation across Nx trajectories
Use many trajectories and initialize them in the stationary distribution.

'''
Nx = 15000
Nt = 300

T = 1

params_CIR = [6, 2, 1]

CIRp = CIRprocess(params_CIR, T, Nx, Nt)
CIRp.init_state = CIRp.StationaryState()
t_data, x_data_CIR = CIRp.ExactSolution()

In [ ]:
n = np.random.randint(0, Nt)
fig, ax = plt.subplots()
x_data = np.linspace(0, 8, 250)

i = np.random.randint(0, Nt)

n, bins, patches = ax.hist(x_data_CIR[i], bins=x_data, density=True, color='#0504aa', alpha=0.7, rwidth=0.85, label = 'SDE sample')
ax.set_xlabel('x')
ax.set_ylabel('p(x)')
ax.set_title(f'CIR-process. Histogram at time step {i}/{Nt}. N = {Nx}', loc = 'left')

ax.plot(x_data, CIRp.StationaryDistribution(x_data), label = 'Stationary CIR PDF', color = 'red')
#ax.plot(x_data, CIRp.ProcessDistribution(x_data, T * i / Nt, x0, 0), label = 'Stationary CIR PDF', color = 'red')

ax.plot(x_data, StationaryDistributionKernelEstimator(x_data, x_data_CIR[i]), label = f'Kernel estimation. Scott rule', color = 'green')

d = 0.01
ax.plot(x_data, StationaryDistributionKernelEstimator(x_data, x_data_CIR[i], delta = d),
        label = f'Kernel estimation. delta = {d}', color = 'orange')

ax.legend(loc = 'upper right', fancybox = True, shadow = True)
plt.show()

#Exercise 2. Implement a non-parametric drift estimator

$$
\hat{b}_n(x) = \frac{\sum_{i=0}^{n-1} K\left(\frac{x-X_i}{h_n}\right)(X_{i+1}-X_i)} {\Delta_n \sum_{i=0}^{n-1} K\left(\frac{x-X_i}{h_n}\right)}
$$

and diffusion estimator
$$
\hat{\sigma}^2_n(x) = \frac{\sum_{i=0}^{n-1} K\left(\frac{x-X_i}{h_n}\right)(X_{i+1}-X_i)^2} {\Delta_n \sum_{i=0}^{n-1} K\left(\frac{x-X_i}{h_n}\right)}.$$


for the model from previous item and plot two graphs.
Use the

*  Gaussian kernel $K(u) = \frac{1}{\sqrt{2\pi}} \exp \left(-\frac{1}{2}u^2\right)$,
*  Uniform kernel $K(u)=1/2$,
*  Epanechnikov (parabolic) kernel $K(u)=\frac{3}{4}(1-u^2)$.


In [ ]:
def NormalKernel(z):
    return 1/np.sqrt(2 * np.pi) * np.exp(-1/2 * z**2)

def UniformKernel(z):
    return 0.5 * (np.abs(z) <= 1)

def ParabolicKernel(z):
    return 0.75 * np.maximum(1 - z**2, 0) * (np.abs(z) <= 1)

def DiffusionKernelEstimator(x, x_data, delta = None, method: Literal['normal', 'uniform', 'parabolic'] = 'normal'):
    n = len(x_data)
    m = 1

    hn = 1

    hn = max(np.std(x_data) * n**(-1 / (m + 4)), 1e-8)

    s2 = np.zeros(len(x))

    for j in range(0, len(x)):
        K1 = 0
        K2 = 0
        z = (x[j] - x_data[:-1]) / hn
        if method == 'normal':
            K = NormalKernel(z)
        elif method == 'uniform':
            K = UniformKernel(z)
        elif method == 'parabolic':
            K = ParabolicKernel(z)
        else:
            raise ValueError(f"method = {method} not implemented")

        K1 = np.sum(K * (x_data[1:] - x_data[:-1])**2)
        K2 = np.sum(K)
        if K2 == 0:
            s2[j] = 0
        else:
            s2[j] = K1 / K2

    if delta is None:
        delta = 1

    return s2 / delta

def DriftKernelEstimator(x, x_data, delta = None, method: Literal['normal', 'uniform', 'parabolic'] = 'normal'):
    n = len(x_data)
    m = 1

    hn = 1

    hn = max(np.std(x_data) * n**(-1 / (m + 4)), 1e-8)

    s2 = np.zeros(len(x))

    for j in range(0, len(x)):
        K1 = 0
        K2 = 0
        z = (x[j] - x_data[:-1]) / hn
        if method == 'normal':
            K = NormalKernel(z)
        elif method == 'uniform':
            K = UniformKernel(z)
        elif method == 'parabolic':
            K = ParabolicKernel(z)
        else:
            raise ValueError(f"method = {method} not implemented")

        K1 = np.sum(K * (x_data[1:] - x_data[:-1]))
        K2 = np.sum(K)
        if K2 == 0:
            s2[j] = 0
        else:
            s2[j] = K1 / K2

    if delta is None:
        delta = 1

    return s2 / delta

In [ ]:
Nx = 1
Nt = 100000

T = 1000

x_data = np.linspace(0, 6, 200)

params_CIR = [6, 2, 1]

CIRp = CIRprocess(params_CIR, T, Nx, Nt)
t_data, x_data_CIR = CIRp.ExactSolution()

In [ ]:
sigma_data = CIRp.sigmaxt(x_data, 0, CIRp.params)

plt.plot(x_data, sigma_data, label = 'Diffusion real')


diffke = DiffusionKernelEstimator(x_data, x_data_CIR.flatten(), T/Nt, 'normal')
plt.plot(x_data, np.sqrt(diffke), label = 'Normal Kernel')

#diffke = DiffusionKernelEstimator(x_data, x_data_CIR.flatten(), T/Nt, 'parabolic')
#plt.plot(x_data, np.sqrt(diffke), label = 'Parabolic Kernel')

#diffke = DiffusionKernelEstimator(x_data, x_data_CIR.flatten(), T/Nt, 'uniform')
#plt.plot(x_data, np.sqrt(diffke), label = 'Uniform Kernel')

plt.title("Diffusion estimation")
plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("x")
plt.ylabel("sigma(x)")
plt.grid(True)
plt.show()

In [ ]:
b_data = CIRp.bxt(x_data, 0, CIRp.params)

plt.plot(x_data, b_data, label = 'Drift real')


driftke = DriftKernelEstimator(x_data, x_data_CIR.flatten(), T/Nt, 'normal')
plt.plot(x_data, driftke, label = 'Normal Kernel')

#driftke = DriftKernelEstimator(x_data, x_data_CIR.flatten(), T/Nt, 'parabolic')
#plt.plot(x_data, driftke, label = 'Parabolic Kernel')

#driftke = DriftKernelEstimator(x_data, x_data_CIR.flatten(), T/Nt, 'uniform')
#plt.plot(x_data, driftke, label = 'Uniform Kernel')

plt.title("Drift estimation")
plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("x")
plt.ylabel("b(x)")
plt.grid(True)
plt.show()

# Exercises

## Task 1 — Estimation of the stationary density from an ensemble
1. Simulate a CIR or OU trajectory and estimate the stationary density using Gaussian, uniform and Epanechnikov kernels.
2. Compare the kernel estimates with the theoretical stationary density.
3. Discuss the influence of the kernel and boundary effects.

In [ ]:
def StationaryDistributionKernelEstimator(sample, x_grid, kernel='gaussian'):
    n = len(sample)
    sigma = np.std(sample)
    h = 1.06 * sigma * (n ** (-0.2))

    density = np.zeros_like(x_grid)
    batch_size = 15000

    for i in range(0, n, batch_size):
        batch = sample[i:i+batch_size]
        u = (x_grid[:, None] - batch[None, :]) / h

        if kernel == 'gaussian':
            K = np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)
        elif kernel == 'uniform':
            K = 0.5 * (np.abs(u) <= 1).astype(float)
        elif kernel == 'epanechnikov':
            K = 0.75 * (1 - u**2) * (np.abs(u) <= 1).astype(float)
        else:
            raise ValueError("Unsupported kernel")

        density += np.sum(K, axis=1)

    density /= (n * h)
    return density, h

np.random.seed(42)

params = [0.5, 1.0, 0.5]
T, Nt, Nx = 500.0, 100000, 1

cir = CIRprocess(params, T, Nx, Nt)
t_data, x_data = cir.EulerSolution()

burn_in = 100.0
mask = t_data > burn_in
stationary_sample = x_data[mask, 0][::5]

x_grid = np.linspace(0, np.max(stationary_sample) * 1.1, 500)
theoretical_pdf = cir.StationaryDistribution(x_grid)

plt.figure(figsize=(10, 6))
plt.plot(x_grid, theoretical_pdf, 'k-', linewidth=2.5, label=r'Theoretical density pi(x)')

kernels = {
    'gaussian': 'Gaussian',
    'uniform': 'Uniform',
    'epanechnikov': 'Epanechnikov'
}

for k_type, k_name in kernels.items():
    kde, h = StationaryDistributionKernelEstimator(stationary_sample, x_grid, k_type)
    plt.plot(x_grid, kde, '--', linewidth=2, label=f'{k_name} kernel (Scott h = {h:.3f})')

plt.xlabel('x', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Kernel estimates of the stationary CIR density', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.6)
plt.tight_layout()
plt.show()


np.random.seed(42)
params = [0.5, 1.0, 0.5]
T, Nt, Nx = 500.0, 100000, 1

cir = CIRprocess(params, T, Nx, Nt)
_, x_data = cir.EulerSolution()

burn_in = 100.0
mask = t_data > burn_in
stationary_sample = x_data[mask, 0][::5]

x_grid_full = np.linspace(-0.5, np.max(stationary_sample) * 1.1, 1000)
kde_gaussian, _ = StationaryDistributionKernelEstimator(stationary_sample, x_grid_full, 'gaussian')

plt.figure(figsize=(10, 5))
plt.plot(x_grid_full, kde_gaussian, 'b-', linewidth=2, label='Gaussian kernel')
plt.plot(x_grid, theoretical_pdf, 'k-', linewidth=2.5, label=r'Theoretical density pi(x)')
plt.fill_between(x_grid_full[x_grid_full < 0], kde_gaussian[x_grid_full < 0], color='red', alpha=0.3, label='Nonzero tail for x<0')
plt.title('Gaussian kernel: nonzero tail for x<0', fontsize=10)
plt.legend()
plt.grid(True, alpha=0.6)
plt.xlim(-0.2, 0.4)
plt.show()

**Conclusions.** All three kernel estimators approximate the theoretical stationary density in the well-sampled region. Gaussian kernels are smooth but assign some probability to negative values, although the CIR state space is nonnegative. Differences near the boundary reflect kernel bias and limited local observations. Results depend on sample size and bandwidth.

## Task 2 — Scott's bandwidth rule
1. Compute the bandwidth using Scott's rule, $h_n\propto n^{-1/5}$.
2. Examine the effect of sample size and bandwidth on the kernel density estimator.
3. Plot a 3D surface of the integrated squared error. Here $1/n$ is a sample-size proxy, **not** a temporal sampling interval.

In [ ]:
def kernel_density_estimate(x_data, x_grid, h):
    n = len(x_data)
    density = np.zeros(len(x_grid))

    for i, x in enumerate(x_grid):
        kernel_values = np.exp(-0.5 * ((x - x_data) / h)**2) / (np.sqrt(2 * np.pi) * h)
        density[i] = np.mean(kernel_values)

    return density


def calculate_ise(estimated_density, true_density, x_grid):
    squared_error = (estimated_density - true_density)**2
    ise = simpson(squared_error, x_grid)
    return ise


def scott_rule(n, sigma_hat=1.0):
    return sigma_hat * n**(-1/5)


params = [2.0, 1.0, 0.5]
Nx = 10000
cir = CIRprocess(params, T=10.0, Nx=Nx, Nt=1000)
stationary_data = cir.StationaryState()

x_min = 0.01
x_max = np.percentile(stationary_data, 99.5)
x_grid = np.linspace(x_min, x_max, 500)

true_density = cir.StationaryDistribution(x_grid)
norm_factor = simpson(true_density, x_grid)
true_density = true_density / norm_factor

sigma_hat = np.std(stationary_data)

n_values = [100, 200, 500, 1000, 2000, 5000]
delta_n = [1/n for n in n_values]
h_factors = np.linspace(0.2, 3.0, 30)

print("Sample sizes:", n_values)
print("Inverse sample size 1/n Δn:", [f"{d:.4f}" for d in delta_n])
scott_h = [scott_rule(n, sigma_hat) for n in n_values]
print("Scott bandwidths:", [f"{h:.4f}" for h in scott_h])

ise_matrix = np.zeros((len(n_values), len(h_factors)))

for i, n in enumerate(n_values):
    indices = np.random.choice(len(stationary_data), size=n, replace=False)
    sample_data = stationary_data[indices]
    base_h = scott_rule(n, sigma_hat)

    for j, factor in enumerate(h_factors):
        h = base_h * factor
        estimated_density = kernel_density_estimate(sample_data, x_grid, h)
        ise_matrix[i, j] = calculate_ise(estimated_density, true_density, x_grid)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

delta_n_grid, h_grid = np.meshgrid(delta_n, h_factors)
surf = ax.plot_surface(delta_n_grid, h_grid, np.log10(np.maximum(ise_matrix.T, 1e-16)),
                       cmap='viridis', alpha=0.8)

ax.set_xlabel('Inverse sample size 1/n')
ax.set_ylabel('Bandwidth')
ax.set_zlabel('log10(ISE)')
ax.set_title('ISE versus bandwidth and inverse sample size')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)

ax.invert_xaxis()

plt.tight_layout()
plt.show()

print(f"{'n':<8} {'Δn=1/n':<12} {'h_scott':<12} {'ISE':<12}")
test_n = [100, 200, 1000, 2000, 7500, 10000]
ise_values = []
h_values = []

for n in test_n:
    indices = np.random.choice(len(stationary_data), size=n, replace=False)
    sample = stationary_data[indices]
    h = scott_rule(n, sigma_hat)
    estimated_density = kernel_density_estimate(sample, x_grid, h)
    ise = calculate_ise(estimated_density, true_density, x_grid)
    ise_values.append(ise)
    h_values.append(h)
    print(f"{n:<8} {1/n:<12.6f} {h:<12.6f} {ise:<12.8f}")

**Conclusion.**
The experiment investigates how kernel bandwidth and sample size affect the accuracy of stationary density estimation.

Scott's rule decreases the bandwidth as the sample size increases, allowing the density estimator to capture finer distributional features while maintaining sufficient smoothing.

The estimation accuracy is measured using the Integrated Squared Error (ISE), which quantifies the overall squared difference between the estimated and theoretical probability densities.

The three-dimensional surface illustrates the trade-off between undersmoothing and oversmoothing. Excessively large bandwidths increase estimation error, while intermediate bandwidth values can provide more accurate density estimates.

The inverse sample size, 1/n, is used to represent changes in the number of observations. Since the samples are generated from a stationary distribution, this quantity should not be interpreted as a physical observation time step.

**Overall**, the results highlight the importance of bandwidth selection and sample size in nonparametric density estimation.

## Task 3 — Nonparametric recovery of SDE coefficients
1. Estimate the drift and squared diffusion coefficient of the CIR process using kernel-weighted increments (Florens–Zmirou-type estimators).
2. Compare estimates with the true coefficient functions.
3. Investigate sensitivity to bandwidth, observation horizon, time discretization and the number of trajectories, especially near rarely visited state-space boundaries.

In [ ]:
def DriftKernelEstimator(x_grid, trajectory, dt, h, kernel='gaussian'):
    current = trajectory[:-1]
    increments = trajectory[1:] - trajectory[:-1]

    n_grid = len(x_grid)
    sum_w = np.zeros(n_grid)
    sum_w_inc = np.zeros(n_grid)
    chunk_size = 20000

    for i in range(0, len(current), chunk_size):
        u = (x_grid[:, None] - current[i:i+chunk_size][None, :]) / h
        if kernel == 'gaussian':
            K = np.exp(-0.5 * u**2) / (h * np.sqrt(2 * np.pi))
        elif kernel == 'epanechnikov':
            K = 0.75 * (1 - u**2) * (np.abs(u) <= 1) / h
        elif kernel == 'uniform':
            K = 0.5 * (np.abs(u) <= 1) / h
        else:
            raise ValueError("Unsupported kernel")

        sum_w += np.sum(K, axis=1)
        sum_w_inc += np.sum(K * increments[i:i+chunk_size][None, :], axis=1)

    mask = sum_w > max(1e-10, 1e-3 * np.max(sum_w))
    res = np.full_like(x_grid, np.nan)
    res[mask] = sum_w_inc[mask] / (dt * sum_w[mask])
    return res

def DiffusionKernelEstimator(x_grid, trajectory, dt, h, kernel='gaussian'):
    current = trajectory[:-1]
    increments_sq = (trajectory[1:] - trajectory[:-1])**2

    n_grid = len(x_grid)
    sum_w = np.zeros(n_grid)
    sum_w_inc2 = np.zeros(n_grid)
    chunk_size = 20000

    for i in range(0, len(current), chunk_size):
        u = (x_grid[:, None] - current[i:i+chunk_size][None, :]) / h
        if kernel == 'gaussian':
            K = np.exp(-0.5 * u**2) / (h * np.sqrt(2 * np.pi))
        elif kernel == 'epanechnikov':
            K = 0.75 * (1 - u**2) * (np.abs(u) <= 1) / h
        elif kernel == 'uniform':
            K = 0.5 * (np.abs(u) <= 1) / h
        else:
            raise ValueError("Unsupported kernel")

        sum_w += np.sum(K, axis=1)
        sum_w_inc2 += np.sum(K * increments_sq[i:i+chunk_size][None, :], axis=1)

    mask = sum_w > max(1e-10, 1e-3 * np.max(sum_w))
    res = np.full_like(x_grid, np.nan)
    res[mask] = sum_w_inc2[mask] / (dt * sum_w[mask])
    return res

np.random.seed(42)
params = [0.5, 1.0, 0.5]
T, Nt, Nx = 500.0, 200000, 1
cir = CIRprocess(params, T, Nx, Nt)
t_data, x_data = cir.EulerSolution()
traj = x_data[:, 0]
dt = T / Nt

x_grid = np.linspace(0.01, 2.5, 300)
h_default = 0.15

b_true = params[0] - params[1] * x_grid
sigma2_true = (params[2] * np.sqrt(x_grid))**2

b_est = DriftKernelEstimator(x_grid, traj, dt, h_default)
sigma2_est = DiffusionKernelEstimator(x_grid, traj, dt, h_default)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(x_grid, b_true, 'k-', lw=2.5, label=r'True $b(x)$')
plt.plot(x_grid, b_est, 'r--', lw=2, label=r'Estimate $\hat{b}_h(x)$')
plt.title(r'Drift recovery $b(x)$')
plt.xlabel(r'$x$'); plt.ylabel(r'$b(x)$')
plt.legend(); plt.grid(True, alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(x_grid, sigma2_true, 'k-', lw=2.5, label=r'True $\sigma^2(x)$')
plt.plot(x_grid, sigma2_est, 'r--', lw=2, label=r'Estimate $\hat{\sigma}^2_h(x)$')
plt.title(r'Diffusion recovery $\sigma^2(x)$')
plt.xlabel(r'$x$'); plt.ylabel(r'$\sigma^2(x)$')
plt.legend(); plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

hs = [0.04, 0.15, 0.5]
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for h in hs:
    plt.plot(x_grid, DriftKernelEstimator(x_grid, traj, dt, h), '--', lw=2, label=f'h={h}')
plt.plot(x_grid, b_true, 'k-', lw=2.5, label=r'True')
plt.title(r'Effect of $h$ on drift estimation')
plt.xlabel(r'$x$'); plt.legend(); plt.grid(True, alpha=0.5)

plt.subplot(1, 2, 2)
for h in hs:
    plt.plot(x_grid, DiffusionKernelEstimator(x_grid, traj, dt, h), '--', lw=2, label=f'h={h}')
plt.plot(x_grid, sigma2_true, 'k-', lw=2.5, label=r'True')
plt.title(r'Effect of $h$ on diffusion estimation')
plt.xlabel(r'$x$'); plt.legend(); plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
def analyze_many_cases():

    params = [0.5, 1.0, 0.5]
    x_grid = np.linspace(0.01, 3.0, 300)
    b_true = params[0] - params[1] * x_grid
    h_fixed = 0.15

    scenarios = [

        (500, 200000, 1, "Baseline: many observations"),
        (500, 200000, 1, "T=500"),
        (100, 200000, 1, "T=100"),
        (20,  200000, 1, "T=20"),
        (10,  200000, 1, "T=10"),
        (500, 200000, 1, "Nt=200k"),
        (500, 50000,  1, "Nt=50k"),
        (500, 10000,  1, " Nt=10k"),
        (500, 200000, 1,  "Nx=1"),
        (500, 200000, 10, "Nx=10"),
        (500, 200000, 50, "Nx=50"),
        (20,  200000, 50, "T=20 + Nx=50"),
    ]

    unique_scenarios = []
    seen = set()
    for T, Nt, Nx, title in scenarios:
        key = (T, Nt, Nx)
        if key not in seen:
            seen.add(key)
            unique_scenarios.append((T, Nt, Nx, title))

    n_plots = len(unique_scenarios)
    n_cols = 4
    n_rows = (n_plots + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten()

    for idx, (T, Nt, Nx, title) in enumerate(unique_scenarios):
        np.random.seed(42)
        dt = T / Nt

        cir = CIRprocess(params, T, Nx, Nt)
        _, x_data = cir.EulerSolution()

        if Nx == 1:
            b_est = DriftKernelEstimator(x_grid, x_data[:, 0], dt, h_fixed)
        else:
            b_estimates = []
            for i in range(min(Nx, 20)):
                be = DriftKernelEstimator(x_grid, x_data[:, i], dt, h_fixed)
                b_estimates.append(be)
            b_est = np.nanmean(b_estimates, axis=0)

        ax = axes[idx]
        ax.plot(x_grid, b_true, 'k-', lw=2, label='True b(x)')
        ax.plot(x_grid, b_est, 'r--', lw=1.5, label='Estimate')

        ax.axvspan(0, 0.3, alpha=0.2, color='red', label='Left boundary (x<0.3)')
        ax.axvspan(2.5, 3.0, alpha=0.2, color='blue', label='Right boundary (x>2.5)')

        left_mask = x_grid < 0.3
        right_mask = x_grid > 2.5

        mse_left = np.nanmean((b_est[left_mask] - b_true[left_mask])**2) if np.any(~np.isnan(b_est[left_mask])) else np.inf
        mse_right = np.nanmean((b_est[right_mask] - b_true[right_mask])**2) if np.any(~np.isnan(b_est[right_mask])) else np.inf

        traj_sample = x_data[:, :min(Nx,5)].flatten()
        visits_left = np.sum(traj_sample < 0.3) / len(traj_sample) * 100
        visits_right = np.sum(traj_sample > 2.5) / len(traj_sample) * 100

        ax.set_title(f'{title}\nT={T}, Nt={Nt}, Nx={Nx}\nMSE left={mse_left:.3f}, right={mse_right:.3f}\nvisits: left={visits_left:.1f}%, right={visits_right:.1f}%', fontsize=8)
        ax.set_xlabel('x', fontsize=9)
        ax.set_ylabel('b(x)', fontsize=9)
        ax.legend(fontsize=6, loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-1.5, 1.5)
        ax.set_xlim(0, 3.0)

        if mse_left > 0.1:
            ax.annotate('poor estimate\n(insufficient data)', xy=(0.15, -1.2), xytext=(0.15, -1.2),
                       fontsize=7, color='red', ha='center', fontweight='bold')
        if mse_right > 0.1:
            ax.annotate('poor estimate\n(rare tail events)', xy=(2.75, -1.2), xytext=(2.75, -1.2),
                       fontsize=7, color='blue', ha='center', fontweight='bold')

        if mse_left < 0.05 and mse_right < 0.05:
            ax.annotate('OK', xy=(1.5, 1.2), fontsize=9, color='green', ha='center', fontweight='bold')

    for idx in range(len(unique_scenarios), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Effect of T, Nt, Nx on drift estimates near boundaries\n'
                'Red region = left boundary (x<0.3) — process rarely visits near zero\n'
                'Blue region = right boundary (x>2.5) — rare large values (tail)',
                fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

analyze_many_cases()

**Conclusion.**
The experiment investigates the accuracy of kernel-weighted estimators for the drift and squared diffusion coefficients of the CIR process.

The results demonstrate a clear bias–variance trade-off in bandwidth selection. A small bandwidth produces noisy estimates, particularly in sparsely sampled regions, whereas an excessively large bandwidth introduces substantial smoothing bias. Among the tested values, a moderate bandwidth of h = 0.15 provides the most visually balanced estimates.

The boundary experiments show that estimation accuracy strongly depends on the local availability of observations. Drift estimates are relatively accurate in frequently visited regions but become unstable near the boundaries, especially in the upper tail where observations are extremely rare or absent.

Increasing the observation horizon, sampling frequency, or number of trajectories can improve estimation accuracy, but additional observations are most useful when they provide sufficient coverage of the relevant state-space regions.

Finally, squared-increment estimators recover the squared diffusion coefficient, σ²(x), rather than the diffusion coefficient σ(x) itself.

**Overall**, the experiment highlights the importance of bandwidth selection, state-space coverage, and sampling design in nonparametric estimation of stochastic differential equation coefficients.

## Task 4 — Mixing and ergodicity
1. Demonstrate convergence toward the stationary distribution from several initial conditions.
2. Compare ensemble means, standard deviations and final-time distributions.
3. Examine the influence of initial conditions and discuss mixing. Numerical agreement illustrates, but does not formally prove, ergodicity.

In [ ]:
params = [2.0, 1.0, 0.5]
T = 20.0
Nt = 2000
Nx = 5000
initial_states = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]

cir = CIRprocess(params, T=T, Nx=Nx, Nt=Nt)

print(f"Initial states: {initial_states}")

trajectories = {}
time_points = np.linspace(0, T, Nt + 1)

for x0 in initial_states:
    cir.init_state = x0 * np.ones(Nx)
    t_data, x_data = cir.EulerSolution()
    trajectories[x0] = x_data.astype(np.float32)
    del x_data
    print(f"X0 = {x0:6.1f}: final mean = {np.mean(trajectories[x0][-1]):.4f}, std = {np.std(trajectories[x0][-1]):.4f}")

x_grid = np.linspace(0.01, 8, 400)
true_density = cir.StationaryDistribution(x_grid)
norm_factor = simpson(true_density, x_grid)
true_density = true_density / norm_factor

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
selected_x0 = [0.1, 1.0, 2.0, 5.0, 10.0]
colors = plt.cm.plasma(np.linspace(0, 1, len(selected_x0)))

ax1 = axes[0]

for x0, color in zip(selected_x0, colors):
    x_data = trajectories[x0]

    mean_traj = np.mean(x_data, axis=1)
    std_traj = np.std(x_data, axis=1)

    step = max(1, Nt // 20)
    idx = np.arange(0, Nt + 1, step)

    ax1.plot(time_points, mean_traj, color=color, linewidth=1.5,
             label=f'X0 = {x0:.1f}')

    ax1.errorbar(time_points[idx], mean_traj[idx], yerr=std_traj[idx],
                fmt='none', ecolor=color, alpha=0.3, capsize=2, capthick=0.5)

theta1, theta2, theta3 = params
stationary_mean = theta1 / theta2
ax1.axhline(y=stationary_mean, color='red', linestyle='--', linewidth=2,
           label=f'Stationary mean = {stationary_mean:.1f}')

stationary_std = theta3 * np.sqrt(stationary_mean) / np.sqrt(2 * theta2)
ax1.fill_between(time_points,
                 stationary_mean - stationary_std,
                 stationary_mean + stationary_std,
                 alpha=0.1, color='red')

ax1.set_xlabel('Time t')
ax1.set_ylabel('Process value X(t)')
ax1.set_title('Mean trajectories from different initial states\n(with standard deviation bars)')
ax1.legend(fontsize=8, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, min(10, T)])

ax2 = axes[1]

hist_colors = plt.cm.viridis(np.linspace(0, 1, len(selected_x0)))
bins = np.linspace(0, 8, 50)

for i, (x0, color) in enumerate(zip(selected_x0, hist_colors)):
    x_data = trajectories[x0]
    final_values = x_data[-1, :]

    ax2.hist(final_values, bins=bins, density=True, alpha=0.3,
             color=color, label=f'X0 = {x0:.1f}',
             histtype='stepfilled', edgecolor='black', linewidth=0.5)

ax2.plot(x_grid, true_density, 'k-', linewidth=2.5, label='Stationary density')

ax2.set_xlabel('x')
ax2.set_ylabel('Density')
ax2.set_title(f'Distribution histograms at T={T}\n(convergence toward stationary density)')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 6])
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(2, 3, figsize=(16, 10))
time_snapshots = [0.5, 1.0, 2.0, 3.0, 5.0, T]
snapshot_idx = [int(t / T * Nt) for t in time_snapshots]

for idx_ax, (t_snap, t_idx) in enumerate(zip(time_snapshots, snapshot_idx)):
    ax = axes2[idx_ax // 3, idx_ax % 3]

    for x0, color in zip(selected_x0, hist_colors):
        x_data = trajectories[x0]
        values_at_t = x_data[t_idx, :]
        ax.hist(values_at_t, bins=bins, density=True, alpha=0.25,
                color=color, histtype='stepfilled', edgecolor='black', linewidth=0.3)

    ax.plot(x_grid, true_density, 'k-', linewidth=2, label='Stationary')
    ax.set_xlabel('x')
    ax.set_ylabel('Density')
    ax.set_title(f't = {t_snap:.1f}')
    ax.set_xlim([0, 6])
    ax.grid(True, alpha=0.3)
    if idx_ax == 0:
        ax.legend(fontsize=6)

plt.suptitle('Distribution evolution from different initial states', fontsize=14)
plt.tight_layout()
plt.show()


print(f"\n{'X0':<8} {'Mean(T)':<15} {'Std(T)':<15} {'Mean deviation':<18} {'Std. deviation':<15}")
stationary_mean = theta1 / theta2
stationary_std_theor = theta3 * np.sqrt(stationary_mean) / np.sqrt(2 * theta2)

for x0 in initial_states:
    final_values = trajectories[x0][-1, :]
    mean_T = np.mean(final_values)
    std_T = np.std(final_values)
    mean_dev = abs(mean_T - stationary_mean) / stationary_mean * 100
    std_dev = abs(std_T - stationary_std_theor) / stationary_std_theor * 100
    print(f"{x0:<8.1f} {mean_T:<15.4f} {std_T:<15.4f} {mean_dev:<17.2f}% {std_dev:<14.2f}%")

from scipy.stats import ks_2samp

print(f"\nKolmogorov–Smirnov distances between final-time distributions:")
print(f"Reference: X0 = {initial_states[len(initial_states)//2]}")
reference_dist = trajectories[initial_states[len(initial_states)//2]][-1, :]

for x0 in initial_states:
    test_dist = trajectories[x0][-1, :]
    ks_stat, p_value = ks_2samp(reference_dist, test_dist)
    print(f"X0 = {x0:6.1f}: KS = {ks_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
params = [2.0, 1.0, 0.5]
T = 20.0
Nt = 2000
Nx = 5000
initial_states = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]
cir = CIRprocess(params, T=T, Nx=Nx, Nt=Nt)
print(f"Initial states: {initial_states}")

trajectories = {}
time_points = np.linspace(0, T, Nt + 1)

for x0 in initial_states:
    cir.init_state = x0 * np.ones(Nx)
    t_data, x_data = cir.EulerSolution()
    trajectories[x0] = x_data.astype(np.float32)
    del x_data
    print(f"X0 = {x0:6.1f}: final mean = {np.mean(trajectories[x0][-1]):.4f}, std = {np.std(trajectories[x0][-1]):.4f}")

x_grid = np.linspace(0.01, 8, 400)
true_density = cir.StationaryDistribution(x_grid)
norm_factor = simpson(true_density, x_grid)
true_density = true_density / norm_factor

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
selected_x0 = [0.1, 1.0, 2.0, 5.0, 10.0]
colors = plt.cm.plasma(np.linspace(0, 1, len(selected_x0)))

ax1 = axes[0]

for x0, color in zip(selected_x0, colors):
    x_data = trajectories[x0]

    mean_traj = np.mean(x_data, axis=1)
    std_traj = np.std(x_data, axis=1)

    step = max(1, Nt // 20)
    idx = np.arange(0, Nt + 1, step)

    ax1.plot(time_points, mean_traj, color=color, linewidth=1.5,
             label=f'X0 = {x0:.1f}')

    ax1.errorbar(time_points[idx], mean_traj[idx], yerr=std_traj[idx],
                fmt='none', ecolor=color, alpha=0.3, capsize=2, capthick=0.5)

theta1, theta2, theta3 = params
stationary_mean = theta1 / theta2
ax1.axhline(y=stationary_mean, color='red', linestyle='--', linewidth=2,
           label=f'Stationary mean = {stationary_mean:.1f}')

stationary_std = theta3 * np.sqrt(stationary_mean) / np.sqrt(2 * theta2)
ax1.fill_between(time_points,
                 stationary_mean - stationary_std,
                 stationary_mean + stationary_std,
                 alpha=0.1, color='red')

ax1.set_xlabel('Time t')
ax1.set_ylabel('Process value X(t)')
ax1.set_title('Mean trajectories from different initial states\n(with standard deviation bars)')
ax1.legend(fontsize=8, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, min(10, T)])

ax2 = axes[1]

hist_colors = plt.cm.viridis(np.linspace(0, 1, len(selected_x0)))
bins = np.linspace(0, 8, 50)

for i, (x0, color) in enumerate(zip(selected_x0, hist_colors)):
    x_data = trajectories[x0]
    final_values = x_data[-1, :]

    ax2.hist(final_values, bins=bins, density=True, alpha=0.3,
             color=color, label=f'X0 = {x0:.1f}',
             histtype='stepfilled', edgecolor='black', linewidth=0.5)

ax2.plot(x_grid, true_density, 'k-', linewidth=2.5, label='Stationary density')

ax2.set_xlabel('x')
ax2.set_ylabel('Density')
ax2.set_title(f'Distribution histograms at T={T}\n(convergence toward stationary density)')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 6])
plt.tight_layout()
plt.show()


**Conclusion.**

The experiment investigates the long-term behavior of the CIR process under different initial conditions.

The ensemble mean trajectories demonstrate exponential convergence toward the theoretical stationary mean of 2.0. Processes starting below this level exhibit increasing mean trajectories, while those starting above it exhibit decreasing mean trajectories.

The distribution plots show that the influence of the initial state gradually diminishes. At early observation times, the empirical distributions differ substantially. As time increases, they increasingly overlap and approach the theoretical stationary density.

By T = 20, the empirical distributions from all five initial states closely match the stationary reference, providing numerical evidence of convergence toward a common invariant distribution.

The theoretical stationary mean and variance provide quantitative benchmarks, while comparisons of empirical distributions and two-sample Kolmogorov–Smirnov statistics can help assess differences between simulated ensembles.

These results support the expected mixing behavior of the CIR process. However, convergence of ensemble distributions alone does not constitute a rigorous proof of ergodicity, which additionally concerns the long-run behavior of time averages along individual trajectories.

**Overall**, the experiment demonstrates that the CIR process progressively loses sensitivity to its initial condition and approaches a stable long-run statistical regime.

## Task 5 — Statistical validation
1. Compare the empirical stationary distribution with the theoretical CIR Gamma distribution using a Kolmogorov–Smirnov statistic.
2. Compare theoretical, empirical and KDE-based distribution functions.
3. Estimate the 90th percentile and evaluate the relative error.
4. Discuss the impact of serial dependence on the interpretation of KS p-values.

In [ ]:
def compute_empirical_cdf(sample, x_grid):
    return np.array([np.mean(sample <= x) for x in x_grid])


def compute_kde_cdf(sample, x_grid, h=None):
    if h is None:
        h = 1.06 * np.std(sample) * len(sample)**(-0.2)
    cdf_vals = np.zeros_like(x_grid)
    chunk = 15000
    for i, x in enumerate(x_grid):
        for j in range(0, len(sample), chunk):
            batch = sample[j:j+chunk]
            u = (x - batch) / h
            K = np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)
            cdf_vals[i] += np.sum(K)
        cdf_vals[i] /= (len(sample) * h)
    cdf_vals = np.cumsum(cdf_vals) * (x_grid[1] - x_grid[0])
    cdf_vals /= cdf_vals[-1]
    return cdf_vals


def find_quantile_from_kde(sample, x_grid, q, h=None):
    cdf = compute_kde_cdf(sample, x_grid, h)
    idx = np.argmin(np.abs(cdf - q))
    return x_grid[idx]


def cir_stationary_gamma_params(params):
    theta1, theta2, theta3 = params
    alpha = 2 * theta1 / (theta3**2)
    beta = (theta3**2) / (2 * theta2)
    return alpha, beta

def cir_stationary_pdf(x, params):
    alpha, beta = cir_stationary_gamma_params(params)
    return gamma.pdf(x, a=alpha, scale=beta)

def cir_stationary_cdf(x, params):
    alpha, beta = cir_stationary_gamma_params(params)
    return gamma.cdf(x, a=alpha, scale=beta)

def cir_stationary_quantile(q, params):
    alpha, beta = cir_stationary_gamma_params(params)
    return gamma.ppf(q, a=alpha, scale=beta)

np.random.seed(42)

params = [0.5, 1.0, 0.5]
T, Nt, Nx = 500.0, 200000, 1
cir = CIRprocess(params, T, Nx, Nt)
t_data, x_data = cir.EulerSolution()
traj = x_data[:, 0]
dt = T / Nt

burn_in = 100.0
mask = t_data > burn_in
stationary_sample = traj[mask][::5]

x_grid = np.linspace(0.01, 3.0, 500)

theo_pdf = cir_stationary_pdf(x_grid, params)
theo_cdf = cir_stationary_cdf(x_grid, params)

emp_cdf = compute_empirical_cdf(stationary_sample, x_grid)
kde_cdf = compute_kde_cdf(stationary_sample, x_grid)

alpha, beta = cir_stationary_gamma_params(params)
def theoretical_cdf_func(x):
    return gamma.cdf(x, a=alpha, scale=beta)

# The trajectory observations are serially correlated; use an independent
# stationary ensemble for a standard one-sample KS p-value.
independent_stationary_sample = cir.StationaryState() if cir.Nx >= len(stationary_sample) else gamma.rvs(
    a=alpha, scale=beta, size=len(stationary_sample), random_state=42
)
ks_stat, ks_pvalue = stats.kstest(independent_stationary_sample, theoretical_cdf_func)
ks_stat_trajectory = stats.kstest(stationary_sample, theoretical_cdf_func).statistic

print("Kolmogorov–Smirnov test (independent stationary sample):")
print(f"KS statistic D = {ks_stat:.4f}")
print(f"p-value = {ks_pvalue:.4f}")
print(f"Trajectory KS statistic (descriptive only) = {ks_stat_trajectory:.4f}")
print(f"Decision: {'Do not reject H0' if ks_pvalue > 0.05 else 'Reject H0'}")

q_level = 0.90
theo_quantile = cir_stationary_quantile(q_level, params)
emp_quantile_direct = np.quantile(stationary_sample, q_level)
kde_quantile = find_quantile_from_kde(stationary_sample, x_grid, q_level)

print(f"Comparison of {q_level*100:.0f}%-quantiles:")
print(f"Theoretical:     {theo_quantile:.4f}")
print(f"Empirical:      {emp_quantile_direct:.4f}")
print(f"From KDE:     {kde_quantile:.4f}")
print(f"Relative error (KDE): {abs(kde_quantile - theo_quantile)/theo_quantile*100:.2f}%")

plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.plot(x_grid, theo_pdf, 'k-', lw=2.5, label=r'Theoretical $\pi(x)$ (Gamma)')
kde_pdf, h = StationaryDistributionKernelEstimator(stationary_sample, x_grid, 'gaussian')
plt.plot(x_grid, kde_pdf, 'r--', lw=2, label=f'KDE (h={h:.3f})')
plt.axvline(theo_quantile, color='blue', linestyle=':', lw=1.5, label=f'90% quantile (theoretical)')
plt.axvline(kde_quantile, color='red', linestyle='--', lw=1, label=f'90% quantile (KDE)')
plt.title('Density of the stationary distribution')
plt.xlabel('$x$'); plt.ylabel('Density')
plt.legend(fontsize=9); plt.grid(True, alpha=0.5)

plt.subplot(2, 2, 2)
plt.plot(x_grid, theo_cdf, 'k-', lw=2.5, label='Theoretical CDF (Gamma)')
plt.plot(x_grid, emp_cdf, 'b-.', lw=1.5, label='Empirical CDF')
plt.plot(x_grid, kde_cdf, 'r--', lw=2, label='KDE-derived CDF')
plt.axhline(q_level, color='gray', linestyle=':', lw=1)
plt.axvline(theo_quantile, color='blue', linestyle=':', lw=1.5)
plt.axvline(kde_quantile, color='red', linestyle='--', lw=1)
plt.title('Cumulative distribution function (CDF)')
plt.xlabel('$x$'); plt.ylabel('F(x)')
plt.legend(fontsize=9); plt.grid(True, alpha=0.5)

plt.subplot(2, 2, 3)
diff_cdf = np.abs(emp_cdf - theo_cdf)
plt.plot(x_grid, diff_cdf, 'm-', lw=2, label='|F_emp - F_theo|')
plt.axhline(ks_stat_trajectory, color='red', linestyle='--', lw=1.5, label=f'Trajectory KS: D={ks_stat_trajectory:.4f}')
plt.title('CDF difference (Kolmogorov–Smirnov statistic)')
plt.xlabel('$x$'); plt.ylabel('Difference')
plt.legend(fontsize=9); plt.grid(True, alpha=0.5)

plt.subplot(2, 2, 4)
plt.hist(stationary_sample, bins=80, density=True, alpha=0.6, label='Histogram', range=(0, 3))
plt.plot(x_grid, theo_pdf, 'k-', lw=2.5, label='Theoretical density')
plt.plot(x_grid, kde_pdf, 'r--', lw=2, label='KDE')
plt.axvline(theo_quantile, color='blue', linestyle=':', lw=1.5, label='90% quantile (theoretical)')
plt.title('Histogram vs theoretical density')
plt.xlabel('$x$'); plt.ylabel('Density')
plt.legend(fontsize=9); plt.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()

**Conclusion.**

The experiment evaluates the agreement between simulated observations of the CIR process and its theoretical stationary Gamma distribution.

The kernel density estimate closely follows the theoretical probability density, while the empirical and KDE-derived cumulative distribution functions show strong agreement with the theoretical CDF.

The estimated 90th percentile is close to its theoretical counterpart, indicating that the kernel density estimator captures the upper part of the distribution with reasonable accuracy.

The Kolmogorov–Smirnov discrepancy calculated from the simulated trajectory is approximately 0.0342, indicating a relatively small maximum difference between the empirical and theoretical cumulative distribution functions. Since successive trajectory observations are serially dependent, this statistic is interpreted descriptively rather than through the standard independent-sample KS p-value.

A separate independent stationary sample can be used for a conventional one-sample Kolmogorov–Smirnov test.

**Overall**, the density estimates, cumulative distribution functions, quantile comparison, and histogram provide consistent numerical evidence that the simulated CIR process exhibits the expected stationary Gamma distribution. These findings support the accuracy of the simulation and estimation procedures within the limits of finite-sample statistical validation.